In [1]:
!pip install chromadb sentence-transformers

  Using cached build-1.4.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached importlib_resources-6.5.2-py3-none-any.whl.metadata (3.9 kB)
  Using cached kubernetes-35.0.0-py2.py3-none-any.whl.metadata (1.7 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached anyio-4.12.1-py3-none-any.whl.metadata (4.3 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
 

In [3]:
import chromadb
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
client = chromadb.Client() #creates a local ChromaDB instance
collection = client.create_collection(name="documind_test") #consider this like a table in a database, it will hold all your data

chunks = [
    "The Eiffel Tower is located in Paris, France.",
    "Machine learning is a subset of artificial intelligence.",
    "Python is a popular programming language for data science.",
    "The Louvre museum contains thousands of art pieces.",
    "Neural networks are inspired by the human brain.",
    "Paris is the capital city of France.",
]
embeddings = model.encode(chunks).tolist()
collection.add(
    documents=chunks,
    embeddings=embeddings,
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)
question = "Where is the Eiffel Tower?"
question_embedding = model.encode([question]).tolist()

results = collection.query(
    query_embeddings=question_embedding,
    n_results=3
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2256.83it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
for i, doc in enumerate(results['documents'][0]):
    print(f"Rank {i+1}: {doc}")

Rank 1: The Eiffel Tower is located in Paris, France.
Rank 2: Paris is the capital city of France.
Rank 3: The Louvre museum contains thousands of art pieces.


In [5]:
question2 = "Which city has the famous iron structure?"

question2_embedding = model.encode([question2]).tolist()
results2 = collection.query(
    query_embeddings=question2_embedding,
    n_results=2
)

print("\n\n🔍 Question 2:", question2)
print("(Notice: zero shared words with the answer)")
print("\n📄 Top 2 retrieved chunks:")
for i, doc in enumerate(results2['documents'][0]):
    print(f"\n  Rank {i+1}: {doc}")



🔍 Question 2: Which city has the famous iron structure?
(Notice: zero shared words with the answer)

📄 Top 2 retrieved chunks:

  Rank 1: The Eiffel Tower is located in Paris, France.

  Rank 2: Paris is the capital city of France.


In [1]:

import sys
sys.path.append('../')

from src.embeddings.embedder import (
    get_chroma_client,
    get_or_create_collection,
    ingest_pdf,
    query_collection
)

client = get_chroma_client("../chroma_db")
collection = get_or_create_collection(client, "documind_week1")

pdf_path = "../data/sample_docs/sample-tables.pdf"
summary = ingest_pdf(pdf_path, collection)

print("\n=== INGESTION SUMMARY ===")
for key, value in summary.items():
    print(f"  {key}: {value}")

print("\n=== QUERYING THE PIPELINE ===")

questions = [
    "What is this document about?",
    "What are the table headers?",
    "How many pages does this document have?"
]

for question in questions:
    print(f"\n🔍 Question: {question}")
    results = query_collection(question, collection, n_results=2)
    for i, result in enumerate(results):
        print(f"\n  Rank {i+1} (distance: {result['distance']})")
        print(f"  Type: {result['metadata']['type']}")
        print(f"  Page: {result['metadata']['page']}")
        print(f"  Content: {result['content'][:150]}...")

c:\Users\Asus\anaconda3\envs\flask_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🤖 Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3402.00it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6010.20it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



📄 Ingesting: ../data/sample_docs/sample-tables.pdf

📝 Step 1: Extracting text...
📄 Loading: ../data/sample_docs/sample-tables.pdf
📝 Total pages: 11
  ✅ Page 1: 946 chars extracted
  ✅ Page 2: 815 chars extracted
  ✅ Page 3: 560 chars extracted
  ✅ Page 4: 1111 chars extracted
  ✅ Page 5: 1031 chars extracted
  ✅ Page 6: 982 chars extracted
  ✅ Page 7: 805 chars extracted
  ✅ Page 8: 821 chars extracted
  ✅ Page 9: 903 chars extracted
  ✅ Page 10: 571 chars extracted
  ✅ Page 11: 519 chars extracted

✅ Loaded 11 pages from sample-tables.pdf
  → 25 text chunks

📊 Step 2: Extracting tables...
🔍 Scanning for tables in: sample-tables.pdf
Consider using the pymupdf_layout package for a greatly improved page layout analysis.
  Page 1, Table 1: extracted
  Page 1, Table 2: extracted
  Page 1, Table 3: extracted
  Page 2, Table 1: extracted
  Page 2, Table 2: extracted
  Page 2, Table 3: extracted
  Page 3, Table 1: extracted
  Page 3, Table 2: extracted
  Page 3, Table 3: extracted
  Page 4, 